# Annotate from a YAML configuration (OMERO or OME-Zarr)

Load a YAML configuration and create a training set from it, or open an existing set from
its folder, then annotate with micro-SAM or export images for Cellpose.

- `source: omero` reads images from an OMERO container (connect in step 2).
- `source: zarr` reads OME-Zarr images, local or remote (e.g. the BioImage Archive), with
  no OMERO server. Such sets use `storage: local`. The example below is
  `notebooks/marimo/configs/zarr-s-biad855-example.yaml`.

The same workflow as a marimo notebook: `notebooks/marimo/annotate-from-yaml.py`.

In [ ]:
from pathlib import Path

import pandas as pd

from omero_annotate_ai import (
    TrainingSet,
    create_omero_connection_widget,
    load_config_from_yaml,
)

## 1. Load the configuration

Give a YAML configuration to create a new training set, or the folder of an existing set
to open it.

In [ ]:
path = Path("../../marimo/configs/zarr-s-biad855-example.yaml").expanduser()

set_folder = path if path.is_dir() else None
config = load_config_from_yaml(path / "config.yaml" if set_folder else path)
needs_omero = config.source == "omero" or config.storage == "omero"

print(f"{'Open' if set_folder else 'New'} training set: {config.name}")
print(f"Source:  {config.source}")
print(f"Storage: {config.storage}")
print(
    f"Splits:  {config.training.train_n} / {config.training.validate_n} / "
    f"{config.training.test_n}"
)
print(f"Needs an OMERO connection: {needs_omero}")

## 2. Connect to OMERO (only when the set needs it)

Skip the widget for a set that reads OME-Zarr and is kept locally; run both cells anyway.

In [ ]:
conn_widget = create_omero_connection_widget()
if needs_omero:
    conn_widget.display()

In [ ]:
conn = conn_widget.get_connection() if needs_omero else None
if needs_omero and conn is None:
    raise ConnectionError(
        "Not connected yet: use the widget above, then rerun this cell."
    )

## 3. Create or open the training set

In [ ]:
ts = (
    TrainingSet.open(set_folder, conn)
    if set_folder
    else TrainingSet.create(config, conn)
)
print(f"{len(ts.units())} units in {ts.bundle_dir}")
pd.DataFrame(
    [
        {
            "split": r.category,
            "image": r.image_name,
            "well": r.well,
            "field": r.field,
            "annotated": r.processed,
        }
        for r in ts.units()
    ]
)

## 4. Annotate

**micro-SAM:** napari opens; segment, then close the window. Each image you segmented is
saved.

In [ ]:
saved = ts.annotate(note=None)  # e.g. note="nuclei only, small debris skipped"
print(f"{saved} units saved in {ts.bundle_dir}")

**Cellpose GUI instead:** run the first cell below, annotate the exported images in the
Cellpose GUI, save each mask in the set's `external_output/` folder as
`<image name>_mask.tif`, then run the second cell.

In [ ]:
# exported = ts.prepare_external()
# print(f"{exported} images exported to {ts.bundle_dir / 'annotation_input'}")
# print(f"Save the masks in {ts.bundle_dir / 'external_output'}")

In [ ]:
# collected = ts.collect_masks()
# print(f"{collected} masks saved")

## 5. Close the connection

In [ ]:
if conn is not None:
    conn.close()